<a href="https://colab.research.google.com/github/yc-115/programing-language/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part2_ipynb%E3%80%8D41371211H.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

安裝必要的套件

In [ ]:
!pip install -q google-generativeai

In [ ]:
import gspread # Added for self-containment
from google.colab import auth # Added for self-containment
from google.auth import default # Added for self-containment
from datetime import datetime # Added for self-containment

In [ ]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

from google.colab import userdata
from google import genai

### 步驟 2: 導入函式庫與設定 API 金鑰

設定 Google Sheet 連線

In [ ]:
import gspread # Added for self-containment
from google.colab import auth # Added for self-containment
from google.auth import default # Added for self-containment
from datetime import datetime # Added for self-containment

# Global variables for Google Sheet connection (re-defined here for self-containment of this test cell)
# These should ideally be defined once in cell 9f9fcf48 and that cell executed.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1wiowNsiqESCAZEjU3p6ZFUcSz7nON3Xvk7lrv2rDdU4/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"
REQUIRED_COLUMNS = ["日期", "考試名稱", "科目", "成績"] # Also from cell 9f9fcf48, added '考試名稱'

_gc = None
_ws = None

def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws
    if _gc is None or _ws is None:
        print("--- 正在進行 Google Sheet 身份驗證和連線... ---")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("--- Google Sheet 連線成功。---")
            print(f"預期欄位順序: {REQUIRED_COLUMNS}") # Added for debugging
        except Exception as e:
            print(f"Google Sheet 連線失敗：{e}")
            _gc = None
            _ws = None

In [ ]:
# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

# (可選) 測試 AI 模型
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

1.  **AI learns patterns from data to make decisions or predictions.**
2.  **Computers learning from experience to make smart choices.**


### 定義 AI 摘要函式

In [ ]:
import re # Added for regex operations

def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for date, exam_name, subject, grade in grades: # Updated to include exam_name
        prompt_text += f"日期：{date}, 考試名稱：{exam_name}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        # Remove common Markdown formatting
        summary = re.sub(r'\*{1,3}(.*?)\*{1,3}', r'\1', summary) # Bold/Italic
        summary = re.sub(r'#{1,6}\s*(.*)', r'\1', summary) # Headers
        summary = re.sub(r'^- (.+)', r'\1', summary, flags=re.MULTILINE) # List items
        summary = re.sub(r'^\* (.+)', r'\1', summary, flags=re.MULTILINE) # List items
        summary = re.sub(r'---{3,}', '', summary) # Horizontal rules
        summary = re.sub(r'>\s*(.*)', r'\1', summary) # Blockquotes
        summary = re.sub(r'\[(.*?)\]\(.*\)', r'\1', summary) # Links
        summary = re.sub(r'\n\n+', '\n\n', summary) # Collapse multiple newlines
        summary = summary.strip()
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [ ]:
def process_grades_and_summary(grade_data):
    """
    處理 Gradio 介面傳入的成績，寫入 Google Sheet 並生成 AI 摘要。
    grade_data 預期是 [考試名稱, 科目, 成績] 的列表的列表，例如：[['第一次段考', '國文', 90], ['第一次段考', '英文', 85]]
    """
    global _gc, _ws

    if _ws is None:
        # 如果連線失敗，嘗試重新設定 (可能在 Gradio 介面啟動後才執行)
        setup_gspread(SHEET_URL, WORKSHEET_NAME) # Modified to pass arguments
        if _ws is None:
            return "Google Sheet 未能成功連線，請檢查錯誤訊息並重試。", ""

    if not grade_data:
        return "沒有輸入任何成績，請輸入科目和成績。", ""

    # 準備寫入 Google Sheet 的成績資料，增加日期欄位
    new_grades_for_sheet = []
    today = datetime.now().strftime('%Y-%m-%d')
    for exam_name, subject, grade_str in grade_data: # Updated to unpack exam_name
        try:
            grade = int(grade_str)
            new_grades_for_sheet.append([today, exam_name, subject, grade]) # Updated to include exam_name
        except ValueError:
            return f"科目 '{subject}' 的成績 '{grade_str}' 無效，成績必須是數字。", ""

    try:
        # 將新成績寫入 Google Sheet
        _ws.append_rows(new_grades_for_sheet)
        sheet_message = "成績已成功寫入 Google Sheet。\n"
    except Exception as e:
        sheet_message = f"寫入 Google Sheet 失敗：{e}\n"
        print(f"寫入 Google Sheet 失敗：{e}")
        # 即使寫入失敗，仍嘗試生成 AI 摘要

    # 獲取 AI 摘要
    summary = get_ai_summary(new_grades_for_sheet)

    try:
        # 尋找第一行空白列來寫入 AI 摘要
        next_row = len(_ws.col_values(1)) + 1
        _ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        _ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            # 確保不會寫入太長導致超出單元格限制 (雖然 gspread 會自動換行)
            _ws.update_cell(next_row + i, 3, line)
        sheet_message += "AI 摘要已成功寫入 Google Sheet。"
    except Exception as e:
        sheet_message += f"寫入 AI 摘要到 Google Sheet 失敗：{e}"
        print(f"寫入 AI 摘要到 Google Sheet 失敗：{e}")

    return sheet_message, summary

In [ ]:
# 確保 Google Sheet 連線已經建立或重新建立
setup_gspread(SHEET_URL, WORKSHEET_NAME)

# 準備測試資料
test_grade_data = [
    ["第一次段考", "國文", "85"],
    ["第一次段考", "數學", "78"],
    ["第一次段考", "英文", "92"]
]

print("\n--- 正在執行 process_grades_and_summary 函式單元測試... ---")

sheet_status, ai_summary_output = process_grades_and_summary(test_grade_data)

print("\n--- 函式執行結果 --- ")
print(f"Google Sheet 處理狀態: {sheet_status}")
print(f"AI 摘要:\n{ai_summary_output}")

# 檢查 _ws 是否為 None，判斷 Google Sheet 是否真的連線成功
if _ws is None:
    print("\n注意：Google Sheet 工作表物件 (_ws) 仍為 None，表示連線可能仍有問題。")
else:
    print("\nGoogle Sheet 工作表物件 (_ws) 已成功初始化，連線似乎已建立。")

--- 正在進行 Google Sheet 身份驗證和連線... ---
--- Google Sheet 連線成功。---
預期欄位順序: ['日期', '考試名稱', '科目', '成績']

--- 正在執行 process_grades_and_summary 函式單元測試... ---

--- 正在呼叫 AI 模型生成摘要... ---

--- 函式執行結果 --- 
Google Sheet 處理狀態: 成績已成功寫入 Google Sheet。
AI 摘要已成功寫入 Google Sheet。
AI 摘要:
好的，這是一份根據您提供的成績資料所做的簡單摘要與常見迷思整理，全程不對學生的表現進行任何評價。

---

成績列表簡單摘要

這是一份來自 2026 年 4 月 2 日 的「第一次段考」成績，包含了三個科目的分數：

   國文：85 分
   數學：78 分
   英文：92 分

這份資料客觀地呈現了該學生在特定時間點的這三個學科考試分數。

---

關於成績的常見迷思整理

學業成績是學生學習過程中的一種回饋形式，但圍繞著成績，社會上常有一些普遍的誤解。以下整理幾個常見迷思，旨在提供更全面的視角：

1.  迷思一：成績高低完全等同於能力與學習成效。
       澄清： 成績確實能反映部分學科知識的掌握程度，但它只是在特定時間、特定測驗範圍下的一種評量。影響成績的因素很多，包括考試當天的狀態、題目設計、個人學習風格，甚至考試焦慮等。它無法全面衡量一個人的綜合能力、創造力、解決問題的能力，或對知識的深層理解與應用。

2.  迷思二：只要努力，成績就一定會很高。
       澄清： 努力是學習的基石，但「有效」的努力和學習策略同樣重要。學習不只關乎投入時間，更關乎理解方法、複習技巧、時間管理等。即使非常努力，若方法不對或遇到學習瓶頸，成績可能無法立刻顯現。學習是一個過程，成長曲線因人而異。

3.  迷思三：單次考試成績就能決定未來發展或個人價值。
       澄清： 任何一次考試成績都只是學習旅程中的一個片段記錄。它不應被視為對個人未來潛力或價值的最終判斷。學生的成長是一個動態過程，包含知識的積累、技能的發展、人格的塑造以及面對挫折的韌性。這些綜合素質遠比單一分數更能影響長期發展。

4.  迷思四：只能跟別人比較成績，才能知

定義 Gradio 處理函式

In [ ]:
import gradio as gr

# Function to load existing grades from Google Sheet
def load_existing_grades():
    global _gc, _ws, SHEET_URL, WORKSHEET_NAME, REQUIRED_COLUMNS
    if _ws is None:
        print("--- 嘗試重新連線 Google Sheet... ---")
        setup_gspread(SHEET_URL, WORKSHEET_NAME)
        if _ws is None:
            print("Google Sheet 連線失敗，無法載入現有成績。")
            return [] # Return empty list for Dataframe if connection fails

    try:
        # Get all values from the worksheet
        all_data = _ws.get_all_values()
        if not all_data or len(all_data) < 1: # Check if there are headers and data
            return [] # No data or only headers, return empty

        # Assuming the first row is headers
        headers = all_data[0]
        records = all_data[1:] # Data starts from the second row

        # Ensure the order of columns matches REQUIRED_COLUMNS for consistency
        # And convert data to list of lists
        processed_data = []
        for row in records:
            current_row = []
            for col_name in REQUIRED_COLUMNS:
                try:
                    col_index = headers.index(col_name)
                    current_row.append(row[col_index])
                except ValueError:
                    current_row.append("") # Column not found, append empty string
            processed_data.append(current_row)
        return processed_data
    except Exception as e:
        print(f"載入 Google Sheet 成績失敗：{e}")
        return [] # Return empty list in case of error

# New function to add a row to the grade input Dataframe
def add_grade_row(current_grades):
    if current_grades is None:
        return [["", "", ""]]
    current_grades.append(["", "", ""])
    return current_grades

with gr.Blocks() as demo:
    gr.Markdown("# 成績輸入與 AI 摘要工具")
    gr.Markdown("請在下方的表格中輸入學生的科目和成績，然後點擊『送出』。系統會將資料寫入 Google Sheet 並生成 AI 摘要。")

    with gr.Tabs():
        with gr.Tab("輸入新成績"):
            with gr.Row():
                with gr.Column():
                    grade_input = gr.Dataframe(
                        headers=["考試名稱", "科目", "成績"],
                        value=[["", "", ""]],
                        type="array",
                        row_count=1, # Fixed row count, will be expanded by the button
                        col_count=(3, "fixed"),
                        label="輸入考試名稱、科目與成績" # Updated label
                    )
                    add_row_button = gr.Button("新增一列科目成績") # New button
                    submit_button = gr.Button("送出")

                with gr.Column():
                    sheet_output = gr.Textbox(label="Google Sheet 處理狀態")
                    summary_output = gr.Textbox(label="AI 摘要", lines=15)

            # Bind new button
            add_row_button.click(
                add_grade_row,
                inputs=grade_input,
                outputs=grade_input # Update the dataframe itself
            )

            submit_button.click(
                process_grades_and_summary,
                inputs=grade_input,
                outputs=[sheet_output, summary_output]
            )
        with gr.Tab("查看現有成績"):
            gr.Markdown("## 查看現有成績")
            load_button = gr.Button("載入現有成績")
            existing_grades_display = gr.Dataframe(
                headers=REQUIRED_COLUMNS, # This will now pick up the updated global variable.
                value=[],
                type="array",
                row_count=10,
                col_count=(len(REQUIRED_COLUMNS), "fixed"),
                label="現有成績 (已更新)" # Changed label to force re-execution.
            )
            load_button.click(
                load_existing_grades,
                inputs=[],
                outputs=existing_grades_display
            )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0f1a4bc12c0f560343.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
